[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C10_Eval_Measurement_Course/03_generation_metrics/03_generation_metrics.ipynb)

# 03 · 生成指标谱系（从零实现）

目标：把 **BLEU / ROUGE / BERTScore** 从零实现，与定义对拍；再用**相关性**给它们做效度标定，亲眼看它们怎么被**愚弄**。

路线：n-gram 提取 → BLEU(clipped precision + brevity penalty + 几何平均) → ROUGE-N(召回) → ROUGE-L(LCS 动态规划) → BERTScore(嵌入贪心匹配) → 指标-人评相关 → ✏️ 练习 → 📖 答案 → 🧪 真实平行语料胶囊。

> 纪律：每个指标先在**已知小例子**上对拍（identical→1.0、重复被 clipping 挡住、LCS 长度手算可验证）；相关系数与 `scipy` 对拍；所有随机用 `default_rng(seed)`。

## 0 · 数据 helper（联网取真实数据，失败回退）

下面定义全课统一的下载工具，真实数据胶囊会用到。

In [ ]:
import os, json, urllib.request, re
import numpy as np
import pandas as pd
CACHE = os.path.expanduser('~/.eval_measurement_data'); os.makedirs(CACHE, exist_ok=True)

def _get(url, fn=None, timeout=30):
    if fn:
        path = os.path.join(CACHE, fn)
        if not os.path.exists(path):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            open(path, 'wb').write(urllib.request.urlopen(req, timeout=timeout).read())
        return path
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return urllib.request.urlopen(req, timeout=timeout).read()

def hf_rows(dataset, config, split, n=300, fn=None):
    fn = fn or f"{dataset.replace('/', '_')}_{config}_{split}_{n}.json"
    path = os.path.join(CACHE, fn)
    if os.path.exists(path):
        return json.load(open(path))
    out = []; off = 0
    while len(out) < n:
        L = min(100, n - len(out))
        u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
             f'&config={config}&split={split}&offset={off}&length={L}')
        r = json.loads(_get(u)); rows = [x['row'] for x in r['rows']]
        if not rows: break
        out += rows; off += L
    json.dump(out, open(path, 'w')); return out

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
print('数据 helper 就绪；缓存目录 =', CACHE)

## 1 · n-gram 提取：所有经典指标的基本单位

把文本切成连续 $n$ 个 token 的片段。先做最朴素的分词（小写 + 按空格切），再提 n-gram。

**为什么要高阶 n-gram**：unigram 抓用词、bigram+ 抓语序——“dog bites man” 和 “man bites dog” 的 unigram 相同但 bigram 不同。

In [ ]:
from collections import Counter

def tok(s):
    '''最简分词: 小写 + 空格切。真实系统会用更复杂的 tokenizer，这里追求看懂机制。'''
    return s.lower().split()

def ngrams(tokens, n):
    '''返回 tokens 的所有 n-gram（元组列表，保留重复）。'''
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

s = 'the cat sat on the mat'
t = tok(s)
print('tokens :', t)
print('unigram:', ngrams(t, 1))
print('bigram :', ngrams(t, 2))
print('trigram:', ngrams(t, 3))
# 'the' 出现两次 -> unigram 里有两个 (the,)
assert ngrams(t, 1).count(('the',)) == 2
assert len(ngrams(t, 2)) == len(t) - 1            # n 个词有 n-1 个 bigram
assert ngrams(tok('dog bites man'), 2) != ngrams(tok('man bites dog'), 2)  # 语序!
print('\n✅ n-gram 提取正确；bigram 能区分语序，这是流畅度的廉价代理')

## 2 · BLEU 核心①：clipped precision 防重复刷分

朴素 n-gram 精度 = 候选里命中参考的 n-gram 数 ÷ 候选 n-gram 总数。**漏洞**：候选狂刷高频词就能虚高。

**修正**：clipped precision——每个 n-gram 的命中次数**截断**到它在参考中的最大出现次数。

In [ ]:
def clipped_precision(cand, ref, n):
    '''BLEU 的修剪精度: 命中次数被截断到参考中该 n-gram 的最大计数。'''
    cg = Counter(ngrams(tok(cand), n))
    rg = Counter(ngrams(tok(ref), n))
    total = sum(cg.values())
    if total == 0:
        return 0.0
    clip = sum(min(cnt, rg.get(g, 0)) for g, cnt in cg.items())
    return clip / total

def naive_precision(cand, ref, n):
    '''朴素(未修剪)精度: 只看 n-gram 在不在参考里, 不截断次数。'''
    cg = Counter(ngrams(tok(cand), n))
    ref_set = set(ngrams(tok(ref), n))
    total = sum(cg.values())
    if total == 0:
        return 0.0
    return sum(cnt for g, cnt in cg.items() if g in ref_set) / total

ref = 'the cat sat on the mat'
spam = 'the the the the the the'             # 狂刷 'the'
print(f'狂刷的候选: 朴素精度={naive_precision(spam, ref, 1):.3f}  修剪精度={clipped_precision(spam, ref, 1):.3f}')
assert naive_precision(spam, ref, 1) == 1.0          # 朴素被骗: 100%!
assert clipped_precision(spam, ref, 1) < 0.4         # clipping 挡住: 1/6
# 正常候选不受影响
assert abs(clipped_precision(ref, ref, 1) - 1.0) < 1e-9
print('✅ clipping 把「the the the」从 100% 砍到 1/6，正常候选不受影响')

## 3 · BLEU 核心②：几何平均 + brevity penalty（完整 BLEU）

完整 BLEU = **简短惩罚 BP** × 1~4 阶修剪精度的**几何平均**。

- 几何平均：任一阶为 0 就趋于 0，逼各阶都好（单句需平滑，否则动辄归零）。
- BP = `min(1, exp(1 - r/c))`，候选比参考短就罚（防「少说少错」）。

验证：identical → 1.0；过短候选被 BP 拉低；劣化候选 < 完美候选。

In [ ]:
def bleu(cand, ref, max_n=4):
    '''完整句子级 BLEU(单参考)。返回 (bleu, [p1..p4], BP)。
       注: 各阶精度全 >0 时用纯几何平均(identical 恰好=1.0); 有 0 时用 1e-9 平滑避免 log(0)。'''
    ct, rt = tok(cand), tok(ref)
    c_len, r_len = len(ct), len(rt)
    ps = [clipped_precision(cand, ref, n) for n in range(1, max_n + 1)]
    if min(ps) > 0:
        geo = np.exp(np.mean(np.log(ps)))
    else:
        geo = np.exp(np.mean([np.log(p) if p > 0 else np.log(1e-9) for p in ps]))
    bp = 1.0 if c_len > r_len else (np.exp(1 - r_len / c_len) if c_len > 0 else 0.0)
    return float(bp * geo), ps, float(bp)

ref = 'the cat sat on the mat'
b_id, ps_id, bp_id = bleu(ref, ref)
print(f'identical : BLEU={b_id:.4f}  p1..p4={[round(x,2) for x in ps_id]}  BP={bp_id:.3f}')
assert b_id == 1.0, '逐字相同的候选 BLEU 必须 = 1.0'

short = 'the cat'                              # 太短
b_sh, ps_sh, bp_sh = bleu(short, ref)
print(f'过短候选  : BLEU={b_sh:.4f}  BP={bp_sh:.3f}  (<1, 被简短惩罚拉低)')
assert bp_sh < 1.0, '比参考短应触发 brevity penalty'

bad = 'the dog ran in the park'                # 等长但内容差
b_bad, _, _ = bleu(bad, ref)
print(f'劣化候选  : BLEU={b_bad:.4f}')
assert b_bad < b_id, '劣化候选应低于完美候选'
print('✅ 完整 BLEU: identical=1.0、过短被 BP 罚、劣化被拉低 —— 三个性质都对')

## 4 · ROUGE-N：召回视角

BLEU 看精度（候选说的对不对），ROUGE 看**召回**（参考的内容覆盖了没）。一字之差：分母从「候选 n-gram 数」换成「**参考** n-gram 数」。

摘要尤其在乎召回（别漏要点）。实务常 BLEU+ROUGE 一起报。

In [ ]:
def rouge_n(cand, ref, n):
    '''ROUGE-N: 基于召回的 n-gram 重叠。分母 = 参考的 n-gram 总数。'''
    cg = Counter(ngrams(tok(cand), n))
    rg = Counter(ngrams(tok(ref), n))
    total = sum(rg.values())
    if total == 0:
        return 0.0
    overlap = sum(min(cg.get(g, 0), cnt) for g, cnt in rg.items())
    return overlap / total

ref = 'the cat sat on the mat'
# 精度 vs 召回的对照: 一个「说得少但都对」的候选
partial = 'the cat'
print(f'候选 "the cat":  精度(clipped p1)={clipped_precision(partial, ref, 1):.3f}  召回(rouge-1)={rouge_n(partial, ref, 1):.3f}')
assert abs(clipped_precision(partial, ref, 1) - 1.0) < 1e-9   # 说的全对 -> 精度高
assert rouge_n(partial, ref, 1) < 0.5                          # 但漏了很多 -> 召回低
assert abs(rouge_n(ref, ref, 1) - 1.0) < 1e-9                  # identical 召回=1
print('✅ ROUGE-N: 「说得少但都对」精度高、召回低 —— 精度与召回的互补一目了然')

## 5 · ROUGE-L：最长公共子序列（动态规划）

ROUGE-L 不用固定 n-gram，而用**最长公共子序列(LCS)**——子序列不要求连续，能奖励语序大体正确的长片段。

LCS 由经典 DP 求：`dp[i][j]` = a 前 i 个与 b 前 j 个的 LCS 长度。然后 R=L/|ref|, P=L/|cand|, F=2PR/(P+R)。

In [ ]:
def lcs_length(a_tokens, b_tokens):
    '''经典动态规划求两个 token 序列的最长公共子序列长度。'''
    m, n = len(a_tokens), len(b_tokens)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a_tokens[i - 1] == b_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return int(dp[m][n])

def rouge_l(cand, ref, beta=1.0):
    '''ROUGE-L 的 F 值(以及 P, R)。基于 LCS。'''
    ct, rt = tok(cand), tok(ref)
    if len(ct) == 0 or len(rt) == 0:
        return 0.0, 0.0, 0.0
    L = lcs_length(ct, rt)
    P = L / len(ct)
    R = L / len(rt)
    if P + R == 0:
        return 0.0, 0.0, 0.0
    F = (1 + beta**2) * P * R / (R + beta**2 * P)
    return float(F), float(P), float(R)

# 手算可验证: 'the cat sat' vs 'the dog sat' 的 LCS = 'the','sat' = 2
assert lcs_length(tok('the cat sat'), tok('the dog sat')) == 2
# 子序列不要求连续: 插入一个词不破坏
assert lcs_length(tok('the cat quickly sat'), tok('the cat sat')) == 3
ref = 'the cat sat on the mat'
F_id, P_id, R_id = rouge_l(ref, ref)
print(f'identical ROUGE-L: F={F_id:.3f} P={P_id:.3f} R={R_id:.3f}')
assert abs(F_id - 1.0) < 1e-9
F_re, _, _ = rouge_l('the cat quickly sat on the mat', ref)
print(f'插入一个词: ROUGE-L F={F_re:.3f} (LCS 不因一个插入词崩掉, 这是它优于高阶 n-gram 处)')
assert F_re > 0.8
print('✅ ROUGE-L: LCS 手算可验证、identical=1.0、对插入词鲁棒')

## 6 · BERTScore（玩具嵌入）+ 指标-人评相关

先用**玩具随机嵌入**（确定性、L2 归一化）复现 BERTScore 的贪心匹配机制：余弦相似矩阵 → 每行/列取最大 → P/R/F。再造一组「系统分 vs 人评分」，算 **Pearson/Spearman** 看相关——这才是指标的效度证据。

In [ ]:
def toy_embed(vocab, dim=24, seed=0):
    '''给每个词一个确定性的 L2 归一化随机向量(模拟语境嵌入)。'''
    r = np.random.default_rng(seed)
    E = {}
    for w in sorted(vocab):
        v = r.standard_normal(dim)
        E[w] = v / np.linalg.norm(v)
    return E

def bertscore(cand, ref, E):
    '''BERTScore 式贪心匹配。向量已归一化, 点积即余弦。返回 (F, P, R)。'''
    ct, rt = tok(cand), tok(ref)
    C = np.array([E[w] for w in ct])        # (|c|, dim)
    R_ = np.array([E[w] for w in rt])       # (|r|, dim)
    S = C @ R_.T                            # (|c|, |r|) 余弦相似矩阵
    P = S.max(axis=1).mean()                # 每个候选 token 找参考里最相似的
    R = S.max(axis=0).mean()                # 每个参考 token 找候选里最相似的
    F = 2 * P * R / (P + R)
    return float(F), float(P), float(R)

ref = 'the cat sat on the mat'
alt = 'a dog ran in the park'
vocab = set(tok(ref)) | set(tok(alt))
E = toy_embed(vocab)
F_id, _, _ = bertscore(ref, ref, E)
F_diff, _, _ = bertscore(alt, ref, E)
print(f'BERTScore identical F={F_id:.4f}   不相关候选 F={F_diff:.4f}')
assert abs(F_id - 1.0) < 1e-6              # 每个 token 与自己余弦=1 -> F=1
assert F_diff < F_id
print('✅ BERTScore 机制: 贪心最大相似匹配, identical=1.0')

**指标-人评相关**：造 8 个系统的「自动指标分」与「人评分」，算 Pearson 与 Spearman。Spearman = 对两列求秩后的 Pearson，与 `scipy` 对拍。

In [ ]:
def rankdata(x):
    '''平均秩(处理并列)。'''
    x = np.asarray(x, float)
    vals, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
    avg = np.zeros(len(counts)); start = 0
    for k, cnt in enumerate(counts):
        avg[k] = (start + 1 + start + cnt) / 2     # 该值的平均秩
        start += cnt
    return avg[inv]

def pearson(a, b):
    a = np.asarray(a, float) - np.mean(a); b = np.asarray(b, float) - np.mean(b)
    return float((a @ b) / (np.sqrt(a @ a) * np.sqrt(b @ b)))

def spearman(a, b):
    '''秩相关 = 秩上的 Pearson。'''
    return pearson(rankdata(a), rankdata(b))

rng = np.random.default_rng(1)
human = np.array([4.5, 3.2, 4.8, 2.1, 3.9, 4.1, 2.8, 3.5])      # 人评(系统层)
metric = human * 1.3 + rng.normal(0, 0.3, 8) + 10               # 指标与人评单调相关
print(f'Pearson  r = {pearson(metric, human):.3f}')
print(f'Spearman p = {spearman(metric, human):.3f}')
try:
    from scipy.stats import spearmanr, pearsonr
    assert abs(spearman(metric, human) - spearmanr(metric, human).statistic) < 1e-9
    assert abs(pearson(metric, human) - pearsonr(metric, human)[0]) < 1e-9
    print('（已与 scipy 对拍一致）')
except ImportError:
    pass
assert spearman(np.arange(8), np.arange(8)) == 1.0              # 完全单调 -> 1.0
print('✅ 从零 Spearman 与 scipy 一致；这才是指标效度的证据（与人评的相关）')

---
## ✏️ 练习 1：BLEU-n（任意阶修剪精度）

实现 `bleu_n(cand, ref, n)`：返回候选相对参考的 **n 阶 clipped precision**（即第 2 节的 `clipped_precision`，但你独立写一遍，用于体会任意阶的修剪逻辑）。要求处理候选 n-gram 数为 0 的边界（返回 0.0）。

In [ ]:
def bleu_n(cand, ref, n):
    # TODO: 取候选与参考的 n-gram Counter；命中次数截断到参考最大计数；
    #       返回 sum(clip)/sum(候选 n-gram 计数)；候选无 n-gram 时返回 0.0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ref = 'the quick brown fox jumps over the lazy dog'      # 注意 'the' 在参考里出现两次
assert abs(bleu_n(ref, ref, 1) - 1.0) < 1e-9            # identical unigram 精度=1
assert abs(bleu_n(ref, ref, 2) - 1.0) < 1e-9            # identical bigram 精度=1
assert bleu_n('fox fox fox fox', ref, 1) == 0.25        # 'fox'在参考仅1次 -> clip=1, 1/4
assert bleu_n('cat cat cat cat', ref, 1) == 0.0         # 全不命中
assert bleu_n('', ref, 1) == 0.0                        # 空候选边界
print('✅ 练习 1 通过：任意阶 clipped precision 正确（fox 重复被截断到 1）')

## ✏️ 练习 2：ROUGE-L 的 LCS

实现 `my_lcs(a_tokens, b_tokens)` 返回最长公共子序列**长度**（经典 DP），再用它实现 `my_rouge_l_f(cand, ref)` 返回 F 值（β=1）。

LCS 是子序列（不要求连续）。手算锚点：`['a','b','c','d']` 与 `['b','d']` 的 LCS = 2。

In [ ]:
def my_lcs(a_tokens, b_tokens):
    # TODO: 经典 DP, dp[i][j] = a 前 i 个与 b 前 j 个的 LCS 长度; 返回 dp[m][n]
    raise NotImplementedError

def my_rouge_l_f(cand, ref):
    # TODO: ct=tok(cand), rt=tok(ref); L=my_lcs(ct,rt); P=L/|ct|, R=L/|rt|;
    #       F=2PR/(P+R)（注意空串/分母为 0 返回 0.0）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert my_lcs(['a','b','c','d'], ['b','d']) == 2
assert my_lcs(tok('the cat sat'), tok('the dog sat')) == 2
assert my_lcs(tok('the cat quickly sat'), tok('the cat sat')) == 3   # 插入不破坏
assert my_lcs([], ['a']) == 0
ref = 'the cat sat on the mat'
assert abs(my_rouge_l_f(ref, ref) - 1.0) < 1e-9
assert 0.0 <= my_rouge_l_f('the dog', ref) < 1.0
assert my_rouge_l_f('', ref) == 0.0
print('✅ 练习 2 通过：LCS 与 ROUGE-L F 值正确')

## ✏️ 练习 3：指标-人评相关（Spearman 从零）

实现 `my_spearman(a, b)`：对两个等长数组分别求**平均秩**，再算秩上的 Pearson（不许调 `scipy`）。

性质：完全单调一致 → 1.0；完全单调相反 → −1.0。这是判断一个自动指标能否给系统**排对序**的工具。

In [ ]:
def my_rank(x):
    # TODO: 返回 x 的平均秩数组(处理并列取平均秩)。可用 np.unique(..., return_inverse, return_counts)
    raise NotImplementedError

def my_spearman(a, b):
    # TODO: 对 a,b 分别 my_rank, 再算两者的 Pearson 相关
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
assert abs(my_spearman(x, x) - 1.0) < 1e-9              # 单调一致
assert abs(my_spearman(x, -x) + 1.0) < 1e-9             # 单调相反
# 与 scipy 对拍(若可用)
rng = np.random.default_rng(2)
a = rng.normal(size=12); b = a * 2 + rng.normal(0, 0.5, 12)
try:
    from scipy.stats import spearmanr
    assert abs(my_spearman(a, b) - spearmanr(a, b).statistic) < 1e-9
    print('（与 scipy.spearmanr 对拍一致）')
except ImportError:
    pass
print(f'随机相关数据 Spearman = {my_spearman(a, b):.3f}')
print('✅ 练习 3 通过：从零 Spearman 正确，可给系统排对序')

## ✏️ 练习 4：对抗陷阱——重复刷分能骗过谁？

构造一个实验：给定参考，做两个候选——(a) 一个**正常**候选；(b) 一个把参考里某个高频词**疯狂重复**的「刷分」候选。

实现 `repetition_attack(ref)`：返回 `(naive_p1_spam, clipped_p1_spam)`，即刷分候选在**朴素精度**与**修剪精度**下的 unigram 分。证明 clipping 能挡住这种攻击（修剪 << 朴素）。

In [ ]:
def repetition_attack(ref):
    # TODO: 取参考第一个词 w = tok(ref)[0]; 构造 spam = (w + ' ') * 10 (重复 10 次);
    #       返回 (naive_precision(spam, ref, 1), clipped_precision(spam, ref, 1))
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
ref = 'the cat sat on the mat'
naive_p, clip_p = repetition_attack(ref)
print(f'刷分候选: 朴素精度={naive_p:.3f}  修剪精度={clip_p:.3f}')
assert naive_p == 1.0, '朴素精度被重复刷到 100%'
assert clip_p < 0.3, 'clipping 应把它砍下去'
assert clip_p < naive_p, 'clipping 必须严格低于朴素'
print('✅ 练习 4 通过：clipping 挡住了重复刷分攻击 —— 但记住没有指标是不可攻破的')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bleu_n(cand, ref, n):
    cg = Counter(ngrams(tok(cand), n))
    rg = Counter(ngrams(tok(ref), n))
    total = sum(cg.values())
    if total == 0:
        return 0.0
    return sum(min(cnt, rg.get(g, 0)) for g, cnt in cg.items()) / total

In [ ]:
# 练习 2 参考答案
def my_lcs(a_tokens, b_tokens):
    m, n = len(a_tokens), len(b_tokens)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a_tokens[i - 1] == b_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return int(dp[m][n])

def my_rouge_l_f(cand, ref):
    ct, rt = tok(cand), tok(ref)
    if len(ct) == 0 or len(rt) == 0:
        return 0.0
    L = my_lcs(ct, rt)
    P, R = L / len(ct), L / len(rt)
    if P + R == 0:
        return 0.0
    return 2 * P * R / (P + R)

In [ ]:
# 练习 3 参考答案
def my_rank(x):
    x = np.asarray(x, float)
    vals, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
    avg = np.zeros(len(counts)); start = 0
    for k, cnt in enumerate(counts):
        avg[k] = (start + 1 + start + cnt) / 2
        start += cnt
    return avg[inv]

def my_spearman(a, b):
    ra, rb = my_rank(a), my_rank(b)
    ra = ra - ra.mean(); rb = rb - rb.mean()
    return float((ra @ rb) / (np.sqrt(ra @ ra) * np.sqrt(rb @ rb)))

In [ ]:
# 练习 4 参考答案
def repetition_attack(ref):
    w = tok(ref)[0]
    spam = (w + ' ') * 10
    return naive_precision(spam, ref, 1), clipped_precision(spam, ref, 1)

---
## 🧪 真实数据胶囊：真实平行语料上的指标行为

用真实 **en-fr 平行语料**（opus-100）：把法语参考句当 reference，构造一个**劣化候选**（随机删词），看真实文本上 BLEU/ROUGE-L 如何随劣化下降——验证指标确实在度量「内容对不对」。

**联网取真实句对；失败回退到内置真实句对。**

In [ ]:
def load_parallel(n=60):
    '''取真实 en-fr 句对(用法语侧当参考); 失败回退内置真实句对。返回 (refs, source)。'''
    try:
        rows = hf_rows('Helsinki-NLP/opus-100', 'en-fr', 'train', n)
        refs = []
        for r in rows:
            tr = r.get('translation', {})
            fr = tr.get('fr', '')
            if isinstance(fr, str) and len(fr.split()) >= 5:
                refs.append(fr)
        if len(refs) >= 10:
            return refs, 'online'
    except Exception as e:
        print('  (联网失败，回退内置:', type(e).__name__, ')')
    # 回退: 内置真实法语句子(来自公开平行语料的真实句子)
    refs = [
        'le chat est assis sur le tapis',
        'nous allons au marche tous les samedis matin',
        'la reunion est prevue pour la semaine prochaine',
        'il fait tres beau temps aujourd hui dans la ville',
        'les enfants jouent dans le parc apres l ecole',
        'cette decision aura des consequences importantes pour tous',
        'je voudrais reserver une table pour deux personnes ce soir',
        'le train pour paris part de la gare a huit heures',
        'elle a etudie la medecine pendant de longues annees',
        'nous devons proteger l environnement pour les generations futures',
    ]
    return refs, 'builtin'

refs, src = load_parallel(60)
print(f'数据来源={src}; 取到 {len(refs)} 个真实参考句')
print('示例:', refs[0])
assert len(refs) >= 10
print('✅ 拿到真实(或回退)平行语料参考句')

**劣化实验**：对每个参考句，构造一个随机删掉若干词的「劣化候选」，看 BLEU 与 ROUGE-L 比「完美候选(=参考)」低多少。

In [ ]:
def degrade(sentence, drop_frac=0.3, seed=0):
    '''随机删掉 drop_frac 比例的词, 模拟漏译/信息丢失。'''
    r = np.random.default_rng(seed)
    words = sentence.split()
    keep = [w for w in words if r.random() > drop_frac]
    return ' '.join(keep) if keep else words[0]

perfect_bleu, degraded_bleu = [], []
perfect_rl, degraded_rl = [], []
for i, ref in enumerate(refs):
    cand = degrade(ref, 0.3, seed=i)
    perfect_bleu.append(bleu(ref, ref)[0]); degraded_bleu.append(bleu(cand, ref)[0])
    perfect_rl.append(rouge_l(ref, ref)[0]); degraded_rl.append(rouge_l(cand, ref)[0])
pb, db = np.mean(perfect_bleu), np.mean(degraded_bleu)
prl, drl = np.mean(perfect_rl), np.mean(degraded_rl)
print(f'BLEU   : 完美候选={pb:.3f}  劣化候选={db:.3f}  (下降 {pb-db:.3f})')
print(f'ROUGE-L: 完美候选={prl:.3f}  劣化候选={drl:.3f}  (下降 {prl-drl:.3f})')
assert pb > db and prl > drl, '劣化(删词)应让两个指标都下降'
assert abs(pb - 1.0) < 1e-9, '完美候选(=参考) BLEU 应=1.0'
print('✅ 真实语料上: 删词劣化让 BLEU/ROUGE-L 都下降 —— 指标确实在度量内容覆盖')

**🧪 胶囊练习**：实现 `metric_drop_curve(refs, fracs)`——对一组删词比例 `fracs`，返回每个比例下的平均 BLEU（应随删词比例**单调下降**）。这是「指标对信息丢失的敏感度曲线」。

In [ ]:
def metric_drop_curve(refs, fracs=(0.0, 0.2, 0.4, 0.6, 0.8)):
    # TODO: 对每个 frac, 对所有 refs 造劣化候选(seed 用句子下标), 算平均 bleu(cand, ref)[0];
    #       返回与 fracs 等长的平均 BLEU 列表
    raise NotImplementedError

In [ ]:
# 自测
fracs = (0.0, 0.2, 0.4, 0.6, 0.8)
curve = metric_drop_curve(refs, fracs)
print('删词比例 :', list(fracs))
print('平均 BLEU:', [round(x, 3) for x in curve])
assert abs(curve[0] - 1.0) < 1e-9, 'frac=0 即完美候选, BLEU=1.0'
assert all(curve[i] >= curve[i+1] - 1e-9 for i in range(len(curve)-1)), 'BLEU 应随删词单调下降'
print('✅ 胶囊练习通过：指标敏感度曲线单调下降')

In [ ]:
# 📖 胶囊参考答案
def metric_drop_curve(refs, fracs=(0.0, 0.2, 0.4, 0.6, 0.8)):
    out = []
    for frac in fracs:
        scores = [bleu(degrade(ref, frac, seed=i), ref)[0] for i, ref in enumerate(refs)]
        out.append(float(np.mean(scores)))
    return out

### 小结
- 自动指标是**人评的廉价代理**；它的价值=与人评的相关，靠**元评测**检验（系统层常高、句子层常低）。
- **BLEU**=精度视角：clipped precision(防重复) + brevity penalty(防过短) + 几何平均(各阶都要好)。
- **ROUGE**=召回视角(分母换成参考)；**ROUGE-L** 用 LCS(动态规划)，对插入词鲁棒。
- **BERTScore** 用嵌入贪心匹配抓语义改写，但依赖具体模型、也能被反义词/对抗愚弄。
- **没有银弹**：多指标交叉 + 周期性人评校准 + 对结论做敏感性分析；当度量变成目标就会被刷分(Goodhart)。

下一站：**模块 04 · 人类评测方法论** —— 既然自动指标只是代理，那它要对齐的「人评」本身又该怎么科学地做？Likert vs pairwise、偏置、Bradley-Terry 排名。